In [12]:
import ctypes
import time
import numpy as np
import pandas as pd
from datetime import datetime

import csv, math, os, sys, copy, random, json
from numpy import asarray, savetxt
from scipy.optimize import curve_fit
from scipy.integrate import quad
from scipy.io import loadmat



from numba import njit
from line_profiler import LineProfiler

####### image single tweezer with camera
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import imageio


import labrad
from labrad.units import WithUnit
from json import JSONEncoder


# %load_ext autoreload
# %autoreload 2  # automatically reloads changes made to labrad_helpers.py
import sys, os
def add_path_up(levels=2):
    target = os.path.abspath(os.path.join(os.getcwd(), *['..']*levels))
    if target not in sys.path:
        sys.path.append(target)
    print("Added to path:", target)


add_path_up(3)


from labrad_helpers import (
    _parseInstructions,
    _engageCamera,
    _initCamera,
    _waitPvcam,
    _setUpCameras,
    jsonize
)


Added to path: c:\Users\Cryo_rdyberg\Documents\Codebase\Development\cam\Tweezer_array2025\ExpSequence


In [30]:
#### load experimental parameters, need to rerun for reloading after change (does not require restarting kernel)
def reload_config():
    import importlib, Experiment_config_v18_6_1210 as Experiment_config_new_seq
    importlib.reload(Experiment_config_new_seq)
    globals().update(vars(Experiment_config_new_seq))

reload_config()

In [31]:
##overide parameters
uv_delay_time_list = [1e-6]
# release_duration_list= np.arange(0,30,5)*1e-6
# uv_duration_list = release_duration_list
#uv_duration_list=np.array([20])*1e-6


#print(tweezer_off_delay_list)
print(repeat_num)
print(uv_freq)
print(uv_duration_list)
print(uv_delay_time_list)

print(folder_path)
print(op_pump_time)
#assert(pgc_cooling_time>=exposure_time)

folder_name = "pvcam"

# Full path to target folder
target_path = os.path.join(folder_path, folder_name)

# Check and create if needed
if not os.path.exists(target_path):
    os.makedirs(target_path)
    print(f"Created folder: {target_path}")




80
90000000.0
[0.0e+00 2.0e-07 4.0e-07 6.0e-07 8.0e-07 1.0e-06 1.2e-06 1.4e-06 1.6e-06
 1.8e-06 2.0e-06 2.2e-06 2.4e-06 2.6e-06 2.8e-06]
[1e-06]
C:\Users\Cryo_rdyberg\Princeton Dropbox\Yukai Lu\CryoRydberg\Data\2025\12\10\lifetime_scan
0.002


In [174]:
op_depump_time = 0#4e-3
#op_depump_time_list = [30e-3]
op_pump_time, op_depump_time, op_amp,bias_xv_for_op_quant_axis,bias_zv_for_op_quant_axis,tweezer_op_amp

(0.002, 0, -47, 3.5, 0.42, 0.02586206896551724)

#### parameters to scan

In [175]:
# # # uv_duration=1e-6
# with labrad.connect() as cxn: 
#     cxn.pulser.freqhop("DDS6", WithUnit(89.88,"MHz")) #87.963#89.963#87.13

In [36]:
#op_depump_time_list = [2e-3]
pushout_time_afteruv = 0
#bias_xv_for_op_quant_axis_list = np.arange(3, 4, 0.1) #[3.5]
op_pump_time = 3e-3 # 100e-6
#op_pump_time_list = [3e-3 ]#[2e-3]#[1e-3, 2e-3, 1e-3]#[2e-3]#[1e-3, 500e-6, 4e-3, 2e-3]#0#2e-3#2e-3 #100e-6#0# 80e-3
op_depump_time = 0
#op_depump_time_list =  [1e-3]#np.array([0, 30e-3])#[0]#[15e-3]#
#op_depump_time_list = [0]#np.array([0, 5e-3, 10e-3, 20e-3, 30e-3, 100e-3])
#op_pump_time_list = np.array([0, 10e-6, 30e-6, 100e-6, 300e-6, 1e-3])
push_out_phase_mode = 'fastpush'#"release_recap"#' '# 'OPtest'#'fastpush'#'release_recap'#'OPtest' #
# push_out_time_list = np.array([0, 1, 1.5, 2, 2.5, 3, 3.5])*1e-6 #np.arange(0, 80, 16)*1e-6 #np.array([])*1e-6 #[32e-6]#[100e-6]#np.array([0, 4, 8, 16, 48])*1e-6#np.arange(0, 80, 16)*1e-6
#push_out_time = #2e-3#48e-6
#extra_idling_time_list = np.array([20])*1e-3
img_field_coolingdetune_for_pgc2_list = [img_field_coolingdetune_for_pgc2]

uv_phase_mode = "two_pi_pulse"#"single_pi_pulse"#"two_pi_pulse"#"single_pi_pulse""two_pi_pulse"#'two_pi_pulse'#"single_pi_pulse"#

#depump_amp_list = [-46]# [-47.8, -47.9]#[-46, -48]
extra_idling_time = 0e-3
push_out_time_list=[2e-6]#np.arange(0,100,12)*1e-6
#uv_duration_list = uv_duration_list[:10]
#push_out_time = 3e-6
#push_out_time_list= np.arange(0, 80, 32)*1e-6
#tweezer_recap_amp_list = [0.69,0.6,0.5,0.3,0.1]
coolingdetune_for_fast_pushout_list = [139.77 -  5.22/64*2.9- 238/64]
uv_freq_list =  [90.50e6]#91.2e6 - np.arange(0, 1.9, 0.1)*1e6#[][89.88e6]#+ np.arange(-0.2, 1, 0.1)*1e6#[87.13e6]#88e6 - np.arange(0, 2, 0.1)*1e6#[90.01e6]#[89.963e6]#- np.arange(-1, 1, 0.1)*1e6#90.5e6 - np.arange(0, 1, 0.1)*1e6#93e6 - np.arange(0, 6, 0.3)*1e6#83.5e6 + np.arange(0, 6, 0.4)*1e6#[5.0775e+08]#[635.5e6]#[635.44e6] - np.arange(-1, 1, 0.1)*1e6
uv_half_pi_time = 0#110*1e-9
uv_half_pi_time_list = np.array([110])*1e-9#,190,200,210,220,230])*1e-9
uv_pi_time_list = [510e-9]#np.arange(460, 560, 10)*1e-9#np.array([510, 520])*1e-9# [510e-9]#np.arange(510, 560, 10)*1e-9#np.array([520])*1e-9#np.arange(400, 480, 20)*1e-9 # np.array([600, 620, 640, 660, 680, 580, 560, 540])*1e-9
uv_pi_time = 510e-9#520e-9#260*1e-9#320*1e-9
uv_duration_list = [0.4e-6]#[1e-6]#[0.4e-6]#np.arange(1,10)*160*1e-9#[520e-9]#[1e-6]#np.arange(10,20)*320*1e-9#[500e-9]##[0.5e-6]# np.arange(225,255, 5)*1e-9#[uv_half_pi_time*2+uv_pi_time]#np.arange(0, 2000, 80)*1e-9#[283e-9] #[290e-9, 280e-9, 300e-9]#np.array([273, 278, 283, 288, 293]*2)*1e-9#[300e-9]#np.array([303, 283, 323, 313, 293]*2)*1e-9 #[uv_pi_time]#np.arange(290, 312, 2)*1e-9#np.arange(1600, 3200, 80)*1e-9# [uv_pi_time]#[600e-9]#np.arange(800, 1400, 80)*1e-9# [uv_pi_time]# np.arange(0, 2000, 200)*1e-9#[600e-9]#np.array([0, 100])*1e-9#np.arange(0, 3000, 400)*1e-9
#uv_duration_list = uv_duration_list[:10]
uv_pulse_separation_list = np.array([2, 12, 13, 4, 6, 8 , 10]*8)*1e-6#np.array([2, 4, 8, 12, 16, 20]*8)*1e-6
uv_pulse_separation = 2e-6
uv_ramsey_idle_time_list = [0]#np.arange(0.35, 5, 0.1)*1e-6
#np.array([0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.95,1.0,1.05,1.1,1.15])*1e-6
tweezer_extra_off_time = 1e-6

#efieldvy = 0.9#0.71#0.447
#efieldvx = -2.1#-1.5#-0.998
#efieldvz = 0.84#1
efieldvy_scan_list = [efieldvy]#[0.76]# [0.92, 0.84, 0.76, 0.7 ]

efieldvy_scan_list, uv_freq_list, uv_pi_time_list , uv_pulse_separation_list, push_out_time_list ##coolingdetune_for_op,op_depump_time_list#tweezer_op_amp_list##,op_depump_time_list
#op_depump_time_list
#op_pump_time_list=[25e-6]
#uv_pi_time_list
#op_pump_time_list
#op_depump_time=1e-3

([0.87],
 [90500000.0],
 [5.1e-07],
 array([2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05,
        2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05,
        2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05,
        2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05,
        2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05,
        2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05,
        2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05,
        2.0e-06, 1.2e-05, 1.3e-05, 4.0e-06, 6.0e-06, 8.0e-06, 1.0e-05]),
 [2e-06])

In [27]:
efieldvy ,efieldvx ,efieldvz

(0.87, -1.55, 0.84)

In [22]:
op_pump_time,op_depump_time, uv_pi_time, uv_half_pi_time, uv_pulse_separation

(0.003, 0, 5.1e-07, 0, 2e-06)

In [28]:
for uv_pulse_separation in uv_pulse_separation_list:
    uv_release_time = tweezer_extra_off_time + (uv_pi_time + uv_half_pi_time*0 + uv_pulse_separation)*1 + uv_pi_time + uv_half_pi_time*0
    uv_release_time = np.max([uv_release_time, 16e-6])
                            #uv_release_time = np.max([uv_release_time, 4e-6])
                            
                            
    print(np.ceil((uv_release_time)*1e6 / 4) * 4/1e6)#uv_pi_time + pi_pulse_separation + uv_pi_time

1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05
1.6e-05


#### 2. Write the sequence

In [ ]:


#assert(image_delay>exposure_time) ### refresh camera during the delay before taking images
with labrad.connect() as cxn:

    ao = cxn.aoserver
    ttl = cxn.finitedopulses
    pulser = cxn.pulser
    #intialization part, does not need to run while scanning or repeating a sequence
    exposure_time = 60e-3
    cxn.pvcamNuvuRfsocServer.initPvCam([ROIleft,ROItop,ROIwidth,ROIheight],WithUnit(exposure_time, 's')) #ROI: [x0,y0,len,len] and exposure time

    #clear previous sequence
    cxn.pvcamNuvuRfsocServer.clearSequence()
    #image status decide if the previous programmed sequence is finished or not
    imageStatus = cxn.pvcamNuvuRfsocServer.isSequenceFinished()
    #@@@@a timer for 60s
    imageWaitIndex = 0
    while (imageStatus == 0 or imageWaitIndex > 60):
        time.sleep(1)
        imageStatus = cxn.pvcamNuvuRfsocServer.isSequenceFinished()
        imageWaitIndex += 1
        print('wait {:.1f}s for image to finish'.format(imageWaitIndex))
    if imageStatus == False:
        raise Warning('pvcamNuvuRfsocServer might have stuck. Check the server')

    #Here will need to change between cycles as some of the parameter (e.g. timing can change)
    #but I think a lot of time the inst_lst is the same, so it is also possible to put it in the previous sec

    #generate a instruction list: simplest now set to just take one image and save it somewhere
    ### @@@@
    inst_lst = _parseInstructions()

    if type(inst_lst) == dict:
        inst_lst = [inst_lst]
    elif type(inst_lst) is not list:
        raise ValueError('Instructions of imaging phase needs to be a list of dicts')
    cxn.pvcamNuvuRfsocServer.acquireImage('pvcam')
    for istr in inst_lst:
        cxn.pvcamNuvuRfsocServer.process(jsonize(istr))

    
    for efieldvy in efieldvy_scan_list:
    #for coolingdetune_for_push_out in coolingdetune_for_push_out_list:
        #print("scan_"+str(push_out_time))
        
        for uv_freq in uv_freq_list:
            #cxn.windfreakserver.freqhop(1, uv_freq)
            cxn.pulser.freqhop("DDS6", WithUnit(uv_freq/1e6,"MHz"))
        #for tweezer_extra_off_time  in tweezer_extra_off_time_list:
        
            #for rydberg_lifetime_duration in rydberg_lifetime_duration_list:
            #for img_field_coolingamp_for_pgc in img_field_coolingamp_for_pgc_list:
            #for coolingdetune_for_img in coolingdetune_for_img_list:
            #for uv_duration in uv_duration_list:
            #    print(uv_duration)
            for uv_pi_time in uv_pi_time_list:
            #for uv_half_pi_time in uv_half_pi_time_list:
            #for op_depump_time in op_depump_time_list:

                #cxn.keysight_33600a.generatePulse(ks_ch, uv_duration, ks_period, ks_vpp, ks_offset, ks_rise_time, ks_fall_time, ks_burst_cycles, "EXT", "INF")
                
                #for img_field_coolingdetune_for_pgc2 in img_field_coolingdetune_for_pgc2_list:
                #for op_depump_time in op_depump_time_list:
                #for recap_ramp_duration in recap_ramp_duration_list:
                #for op_pump_time in op_pump_time_list:
                #for coolingdetune_for_fast_pushout in coolingdetune_for_fast_pushout_list:
                
                #for uv_ramsey_idle_time in uv_ramsey_idle_time_list:
                for uv_pulse_separation in uv_pulse_separation_list:
                #for coolingdetune_for_op in coolingdetune_for_op_list:
                #for depump_amp in depump_amp_list:
                #for bias_yv_for_nulling in bias_yv_for_nulling_list:
                #for coolingamp_for_img in coolingamp_for_img_list:
                    

                #for img_field_coolingamp_for_pgc in img_field_coolingamp_for_pgc_list:
                #for bias_yv_for_nulling in bias_yv_for_nulling_list:
                #for adiabatic_cool_time in adiabatic_cool_time_list:
                #
                #for tweezer_adiabatic_cooling_amp in tweezer_adiabatic_cooling_amp_list:
                    #for tweezer_op_amp in tweezer_op_amp_list:
                    
                    #for op_depump_time in op_depump_time_list:
                    
                    for push_out_time in push_out_time_list:
                    #for tweezer_recap_amp in tweezer_recap_amp_list:
                    #for bias_xv_for_op_quant_axis in bias_xv_for_op_quant_axis_list:
                        #duration in unit of s
                        #op_start_time is used to be recap_time - releaes_time_scan
                        number_of_phase = 11
                        phase_time_list = np.zeros(number_of_phase)

                        j=0
                        phase_time_list[j] = np.ceil((mot_loading_time + delay_after_loading)*1e6 / 4) * 4/1e6
                        j+=1
                        phase_time_list[j] = np.ceil((loading_field_pgc_cooling_time)*1e6 / 4) * 4/1e6
                        
                        j+=1  ### phase 3
                        phase_time_list[j] = np.ceil((tweezer_ramp_down_time + extra_cooling_time + coolingimg_time)*1e6 / 4) * 4/1e6
                        
                        j+=1  ### phase 4: pgc cooling after imaging, only for cooling
                        phase_time_list[j] = np.ceil((img_field_pgc_cooling_time + extra_idling_time)*1e6 / 4) * 4/1e6    

                        j+=1 ### phase 5: OP
                        phase_time_list[j] = np.ceil((op_delay_after_switching_bfield  + op_pump_time + op_depump_time)*1e6 / 4) * 4/1e6
                        
                        j+=1 ### phase 6: Adiabatic cool
                        phase_time_list[j] = np.ceil((adiabatic_cool_time)*1e6 / 4) * 4/1e6

                        j+=1 ## pushout for OPtest
                        #switch different mode
                        #push_out_phase_mode = 'release_recap'#'OPtest' #
                        #phase_time_list[j] = np.ceil(push_out_time*1e6 / 4) * 4/1e6
                        #phase_time_list[j] = np.ceil((np.max([push_out_time,4e-6]))*1e6 / 4) * 4/1e6

                        j+=1
                        if uv_phase_mode == "single_pi_pulse": ###  "single_pi_pulse" "two_pi_pulse"
                            
                            if uv_duration<=0:
                                cxn.keysight_33600a.single_pulse(0,50,0.0,1) #single sine wave for driving: vpp(V),duration(ns),phase(0->2*np.pi), burst_num (default 1)
                            else:
                                #pass
                                cxn.keysight_33600a.single_pulse(0.78,uv_duration*1e9,0.0,1)  ### SET AT 0.78V
                                #uv_duration = uv_half_pi_time*2+uv_ramsey_idle_time
                                #cxn.keysight_33600a.ramsey_pulse(0.78,uv_half_pi_time*1e9,uv_ramsey_idle_time*1e9,1.0,1)
                            uv_release_time = tweezer_extra_off_time + uv_duration#(uv_duration+uv_pulse_separation)*0+uv_duration + pushout_time_afteruv 
                            uv_release_time = np.max([uv_release_time, 4e-6])
                                #cxn.keysight_33600a.single_pulse(0.78,uv_duration*1e9,0.0,1)
                        #assert(uv_extra_delay + rydberg_lifetime_duration + uv_pi_time*2+pushdds_delay_time + push_out_time<uv_release_time)
                            
                        
                        if uv_phase_mode == "two_pi_pulse":
                            #cxn.keysight_33600a.composite_pulse(0.78,uv_half_pi_time*1e9,uv_pi_time*1e9,1) #composite pulse:  vpp(V),uv_pi_time(ns!!!), burst_num (default 1), AOM_rise_delay(ns)
                            #cxn.keysight_33600a.single_tukey_pulse(0.78,uv_pi_time*1e9,0.0,1) 
                            cxn.keysight_33600a.single_pulse(0.78,uv_pi_time*1e9,0.0,1)
                            uv_release_time = tweezer_extra_off_time + (uv_pi_time + uv_half_pi_time*0 + uv_pulse_separation)*1 + uv_pi_time + uv_half_pi_time*0
                            assert(uv_release_time<=24e-6)
                            uv_release_time = np.max([uv_release_time, 16e-6])
                            #uv_release_time = np.max([uv_release_time, 4e-6])
                            
                            
                        phase_time_list[j] = np.ceil((uv_release_time)*1e6 / 4) * 4/1e6#uv_pi_time + pi_pulse_separation + uv_pi_time
                        #print("uv phase release time:", phase_time_list[j])

                        j+=1
                        phase_time_list[j] = np.ceil((recap_hold_duration + recap_ramp_duration)*1e6 / 4) * 4/1e6

                        j+=1
                        phase_time_list[j] = np.ceil((img_tot_time)*1e6 / 4) * 4/1e6
                        j+=1
                        phase_time_list[j]= np.ceil((tweezer_wait_time)*1e6 / 4) * 4/1e6
                        
                        

                        #if we want to disable a phase, we put phase_time_list[i]=0

                        duration = np.round(np.sum(phase_time_list), 6)+0.01 
                        #we add this to round TTL and AO has same length (4us time resolution for AO)
                        duration = np.ceil(duration*1e6 / 4) * 4/1e6
                        
                        
                        #every exp cycle for duration [S]
                        ### Make sure the servers for AO, TTL, camera are open
                        ao.blankwaveform(duration)
                        ttl.blankwaveform(duration)
                        
                        pulser.new_sequence()
                        pulser.line_trigger_state(True)
                        DDS = []
                        t_clock = 0
                        t_clock_list = [] ### t_clock_list[i] with t_clock[0]=0, t_clock[i] recording the end time point of phasei/ start time of phasei+1
                        t_clock_list.append(t_clock)


                        ao.setvoltagepulse(efieldvy_ao, 0, duration, WithUnit(efieldvy,"V"))
                        ao.setvoltagepulse(efieldvz_ao, 0, duration, WithUnit(efieldvz,"V"))
                        ao.setvoltagepulse(efieldvx_ao, 0, duration, WithUnit(efieldvx,"V"))
                        
                        j=0
                        if phase_time_list[j]>0:
                            ################
                            ##### Phase1: load into MOT and tweezer
                            #phase1_time = mot_loading_time + delay_after_loading

                            ####@@@@!!! DDS needs to be fully defined over the sequence
                            DDS.append((offsetlock, WithUnit(0.5, 'us'),  ### 0.5 is a small start time required for using DDS
                                        WithUnit(mot_loading_time-dds_start_time_delay -5e-7, 's'), #### @@@@-5e-7?
                                        WithUnit(coolingdetune_for_loading, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'), 
                                        WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            

                            ttl.PulseOn(tweezer_slm_AOM, 0, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, 0, phase_time_list[j])
                            ao.setvoltagepulse(tweezer_slm_amp, 0, phase_time_list[j], WithUnit(tweezer_loading_amp,"V"))
                            #3D MOT
                            ## Turn on quadrupole B field 
                            ttl.PulseOn(mot_coil, 0, mot_loading_time)


                            pulser.add_ttl_pulse('TTL2', WithUnit(0.1, 'us'), WithUnit(0.1, 'us')) #### @@@??
                            ttl.PulseOn(dds_trigger, 0, 1e-5)

                            # #2D MOT
                            ttl.PulseOn(two_d_motAOM, 0, mot_loading_time)
                            #Repump, for both 2d and 3d mot
                            ttl.PulseOn(repump_motAOM, 0, mot_loading_time)
                            #Push beam controlling flux from 2D MOT
                            ttl.PulseOn(push_beamAOM, 0, mot_loading_time)
                            ## 3d mot cooling beam
                            ttl.PulseOn(three_d_motAOM, 0, mot_loading_time)
                            #ao.setvoltagepulse(offsetlock, 0, mot_loading_time, WithUnit(coolingdetune_for_loading,"V"))  
                            ao.setvoltagepulse(three_d_motAOM_amp, 0, mot_loading_time, WithUnit(coolingamp_for_loading,"V")) 
                            ao.setvoltagepulse(three_d_motAOM_amp, mot_loading_time, delay_after_loading, WithUnit(loading_field_coolingamp_for_pgc,"V")) 

                            
                            ## bias field used for overlappin MOT with tweezers
                            ao.setvoltagepulse(bias_fieldx, 0, mot_loading_time, WithUnit(bias_xv_for_loading,"V"))
                            ao.setvoltagepulse(bias_fieldy, 0, mot_loading_time, WithUnit(bias_yv_for_loading,"V"))

                            
                            biasBz_forward_ttl = 16
                            ttl.PulseOn(biasBz_forward_ttl, 0, mot_loading_time)
                            ao.setvoltagepulse(bias_fieldz, 0, mot_loading_time, WithUnit(bias_zv_for_loading,"V"))###  when TTL is off bias z set by the fixed power supply
                            if reverse_zv_nulling:
                                biasBz_forward_ttl = biasBz_reverse_ttl
                            ttl.PulseOn(biasBz_forward_ttl, mot_loading_time, delay_after_loading)

                            
                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)

                        j+=1
                        if phase_time_list[j]>0:
                            ################
                            ###### Phase2: PGC cooling in deep tweezer
                            #phase2_time = loading_field_pgc_cooling_time

                            DDS.append((offsetlock, WithUnit(mot_loading_time-dds_start_time_delay, 's'), 
                                        WithUnit(delay_after_loading+loading_field_pgc_cooling_time, 's'),  
                                        WithUnit(loading_field_coolingdetune_for_pgc, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'), 
                                        WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))

                            ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])
                            
                            ao.setvoltagepulse(tweezer_slm_amp, t_clock, phase_time_list[j], WithUnit(tweezer_loading_amp,"V"))
                        
                            
                            
                            #### cooling beam applied with some delay after turning off MOT to reduce background, 100ms should be more than enough. 
                            ###At high tweezer we do not want to image, just to lower the temp
                            ttl.PulseOn(three_d_motAOM, t_clock, loading_field_pgc_cooling_time)  ####@@@ vs changed from pgc_cooling_time*2 to pgc_cooling_time
                            ttl.PulseOn(repump_motAOM, t_clock, loading_field_pgc_cooling_time)
                            #ttl.PulseOn(tweezer_camera_trigger, loading_tot_time-6e-4, 1e-5)

                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, loading_field_pgc_cooling_time, WithUnit(loading_field_coolingamp_for_pgc,"V"))
                            
                            ao.setvoltagepulse(bias_fieldx, mot_loading_time, delay_after_loading+loading_field_pgc_cooling_time, WithUnit(bias_xv_for_nulling_high,"V")) ## 
                            #this rampToVoltageTime method is chn,final_voltage, start_time, end_time
                            
                            ao.setvoltagepulse(bias_fieldy, mot_loading_time, delay_after_loading+loading_field_pgc_cooling_time, WithUnit(bias_yv_for_nulling,"V"))
                            ### switch Bz from the one for mot loading to the PGC configuration which is in an opposite direction.
                            # The dt here makes sure Bz is back to the value for loading before starting a new shot.
                            #ttl.PulseOn(biasBz_direction, mot_loading_time, duration-mot_loading_time-dt) 
                            ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, mot_loading_time, delay_after_loading+loading_field_pgc_cooling_time, WithUnit(bias_zv_for_nulling,"V")) ## 

                            
                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)
                        
                        j+=1

                        if phase_time_list[j]>0:
                            ################
                            ###### Phase3: lower tweezer trap depth to an intermediate depth and image with PGC config
                            #phase3_time = tweezer_ramp_down_time + coolingimg_time 
                            
                            DDS.append((offsetlock, WithUnit(t_clock -dds_start_time_delay, 's'), ### @@@ dds start time delay
                                        WithUnit(phase_time_list[j], 's'),  #### @@@?? WithUnit(tweezer_ramp_down_time*2+coolingimg_time, 's'), why *2??
                                        WithUnit(coolingdetune_for_img, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'),
                                        WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])

                            ao.rampToVoltageTime(tweezer_slm_amp, WithUnit(tweezer_img_amp,"V"),t_clock, t_clock + tweezer_ramp_down_time)
                            ao.setvoltagepulse(tweezer_slm_amp,
                                            t_clock + tweezer_ramp_down_time, 
                                            extra_cooling_time + coolingimg_time, WithUnit(tweezer_img_amp,"V"))
                            
                            
                            
                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))
                            #DDS.append((offsetlock, WithUnit(mot_loading_time, 's'), WithUnit(duration-mot_loading_time-dt, 's'), WithUnit(coolingdetune_for_img, 'MHz'), WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            ### change the cooling light detuning and amp right after turning off MOT
                            #ao.setvoltagepulse(offsetlock, mot_loading_time, duration-mot_loading_time-dt, WithUnit(coolingdetune_for_pgc, "V")) 
                            ##### in format (channel num, start time, duration, voltage), The voltage corresponds to -10MHz/V i.e., -2Gamma/V for voltage between -7 and 7V.


                            ######second cooling with lowered trap, image simultaneously
                            ttl.PulseOn(three_d_motAOM, t_clock + tweezer_ramp_down_time, extra_cooling_time + exposure_time) ####@changed from pgc_cooling_time*2 to pgc_cooling_time
                            ttl.PulseOn(repump_motAOM, t_clock + tweezer_ramp_down_time, extra_cooling_time + exposure_time)
                            ttl.PulseOn(tweezer_camera_trigger, t_clock + tweezer_ramp_down_time+extra_cooling_time -tweezer_cam_delay, 1e-5)


                            
                            
                            ao.rampToVoltageTime(bias_fieldx,
                                                WithUnit(bias_xv_for_nulling,"V"),
                                                t_clock,
                                                t_clock + tweezer_ramp_down_time)
                            ao.setvoltagepulse(bias_fieldx, t_clock + tweezer_ramp_down_time, extra_cooling_time + coolingimg_time, WithUnit(bias_xv_for_nulling,"V")) ##
                            ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                            ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(bias_zv_for_nulling,"V")) ## 

                            
                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)
                            
                            ################
                            ###### Phase4: extra short PGC with further detuning in the lower depth
                            #phase3_time =  img_field_coolingamp_for_pgc 4ms
                        j+=1
                        if phase_time_list[j]>0:
                            # DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'), 
                            #             WithUnit(phase_time_list[j], 's'), 
                            #             WithUnit(img_field_coolingdetune_for_pgc, 'MHz'), 
                            #             WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                            
                            DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'), 
                                        WithUnit(img_field_pgc_cooling_time, 's'), 
                                        WithUnit(img_field_coolingdetune_for_pgc, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            
                            DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay+img_field_pgc_cooling_time, 's'), 
                                        WithUnit(phase_time_list[j] - img_field_pgc_cooling_time, 's'),
                                        WithUnit(img_field_coolingdetune_for_pgc2, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            
                            
                            ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])

                            ao.setvoltagepulse(tweezer_slm_amp, t_clock, phase_time_list[j], WithUnit(tweezer_img_amp,"V"))
                            
                            
                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(img_field_coolingamp_for_pgc,"V"))

                            ######second cooling with lowered trap, image simultaneously
                            ttl.PulseOn(three_d_motAOM, t_clock, img_field_pgc_cooling_time) #phase_time_list[j]) ####@changed from pgc_cooling_time*2 to pgc_cooling_time
                            ttl.PulseOn(repump_motAOM, t_clock,  img_field_pgc_cooling_time) #phase_time_list[j])

                            ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_nulling,"V")) ##
                            ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                            ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(bias_zv_for_nulling,"V")) ## 

                                                        
                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)

                        j+=1
                        
                        if phase_time_list[j]>0:
                            ###############
                            ##### Phase 5: optical pumping
                            assert(op_delay_after_switching_bfield>=tweezer_ramp_down_time)
                            # phase4_time = op_delay_after_switching_bfield  + op_pump_time
                            DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'),
                                        WithUnit(phase_time_list[j], 's'),
                                        WithUnit(coolingdetune_for_op, 'MHz'),
                                        WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                            ttl.PulseOn(OP_dds_switch, t_clock, phase_time_list[j])
                            DDS.append((OP_dds, WithUnit(t_clock+ op_delay_after_switching_bfield -dds_start_time_delay, 's'), 
                                    WithUnit(op_pump_time, 's'), 
                                    WithUnit(opAOM_freq, 'MHz'), 
                                    WithUnit(op_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                            DDS.append((op_repump_dds, WithUnit(t_clock+ op_delay_after_switching_bfield -dds_start_time_delay, 's'), 
                                    WithUnit(op_pump_time, 's'),
                                    WithUnit(op_repump_aom_freq, 'MHz'),
                                    WithUnit(op_repump_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            

                            if op_depump_time>0:
                                DDS.append((OP_dds, WithUnit(t_clock+ op_delay_after_switching_bfield +op_pump_time -dds_start_time_delay, 's'), 
                                        WithUnit(op_depump_time, 's'), 
                                        WithUnit(opAOM_freq, 'MHz'), 
                                        WithUnit(depump_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                                                     

                            ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])
                            ao.rampToVoltageTime(tweezer_slm_amp,
                                                WithUnit(tweezer_op_amp,"V"),
                                                t_clock , t_clock + tweezer_ramp_down_time)
                            
                            ao.setvoltagepulse(tweezer_slm_amp, t_clock + tweezer_ramp_down_time, phase_time_list[j] - tweezer_ramp_down_time, WithUnit(tweezer_op_amp,"V"))


                            #ao.setvoltagepulse(offsetlock, loading_tot_time+2*pgc_time, op_tot_time, WithUnit(coolingdetune_for_op,"V")) ##
                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))
                            #### Turn on OP and repump, leave OP on longer for depump if not dark
                            #### @@@@1013 turn off op for test release recap
                            #ttl.PulseOn(opAOM, t_clock + op_delay_after_switching_bfield, op_pump_time)
                            #if op_pump_time>0:

                            #ttl.PulseOn(repump_motAOM, t_clock + op_delay_after_switching_bfield, op_pump_time)

                            #### apply bias field
                            #bias_yv_for_op_quant_axis is now set using another external power supply set at 4.43V.
                            #bias_fieldy ao channel is not affecting anything in this phase.  We keep it unchanged at nulling amp for stability.
                            ttl.PulseOn(bias_y_switch_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                            ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_op_quant_axis,"V"))
                            
                            #if reverse_zv_op_nulling:
                            if bias_zv_for_op_quant_axis <0:
                                ttl.PulseOn(biasBz_reverse_ttl, t_clock, phase_time_list[j])
                            else:
                                ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(np.abs(bias_zv_for_op_quant_axis),"V"))
                            
                            
                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)

                        j+=1
                        if phase_time_list[j]>0:
                            ################
                            ###### Phase6: adiabatic cooling by ramping down tweezer depth 
                            #phase5_time = adiabatic_cool_time
                            #we jump freq here to save time
                            DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'), 
                                        WithUnit(phase_time_list[j], 's'), 
                                        WithUnit(coolingdetune_for_fast_pushout, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            #DDS.append((OP_dds, WithUnit(t_clock -dds_start_time_delay, 's'), 
                            #            WithUnit(phase_time_list[j], 's'),
                            #            WithUnit(opAOM_shut_freq, 'MHz'),
                            #            WithUnit(opAOM_shut_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                            #DDS.append((op_repump_dds, WithUnit(t_clock -dds_start_time_delay, 's'),
                            #            WithUnit(phase_time_list[j], 's'), 
                            #            WithUnit(op_repump_aom_freq, 'MHz'),
                            #            WithUnit(opAOM_shut_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            
                             
                            ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])

                            if push_out_phase_mode == 'OPtest':
                                ao.rampToVoltageTime(tweezer_slm_amp,
                                                WithUnit(tweezer_op_amp,"V"),
                                                t_clock, t_clock + phase_time_list[j])
                            else:
                                ao.rampToVoltageTime(tweezer_slm_amp,
                                                WithUnit(tweezer_adiabatic_cooling_amp,"V"),
                                                t_clock, t_clock + phase_time_list[j])
                            

                            #ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_pgc,"V"))                        
                            #depreciated bias_yv_for_op_quant_axis as we now use TTL for switching. We keep the nulling amp to make power supply more stable when switching
                            ttl.PulseOn(bias_y_switch_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                            ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_op_quant_axis,"V")) 
                            if bias_zv_for_op_quant_axis <0:
                                ttl.PulseOn(biasBz_reverse_ttl, t_clock, phase_time_list[j])
                            else:
                                ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(np.abs(bias_zv_for_op_quant_axis),"V"))

                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)



                        j+=1
                        if phase_time_list[j]>0:
                            print("pushoutphase time", phase_time_list[j])
                            ################
                            ###### Phase7: Turn off trap, push out beam
                            ###note that here might be confusing that we change the DDS freq in the last phase for saving the time
                            ###for this phase we have different configuration
                            ###Push all atoms in ground states: turn on OPaom and Repump with full power, turn off tweezers, aim to push away all atoms in 2us
                            ###Push atoms in the F=4, only turn on OPaom with weak power(same as OP), turn on tweezer, aim to push away atoms in 100us
                            ###No push beam, used when doing release recap
                            
                            if push_out_phase_mode == 'OPtest':
                                ttl.PulseOn(OP_dds_switch, t_clock, phase_time_list[j])
                                DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'), 
                                            WithUnit(phase_time_list[j], 's'), 
                                            WithUnit(coolingdetune_for_op_pushout, 'MHz'), 
                                            WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB'))) 
                                DDS.append((OP_dds, WithUnit(t_clock -dds_start_time_delay, 's'), 
                                            WithUnit(push_out_time, 's'), 
                                            WithUnit(opAOM_freq, 'MHz'), 
                                            WithUnit(push_out_amp_OPtest, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                                                        
                                #no repump
                                #ttl.PulseOn(repump_motAOM, t_clock, push_out_time)

                                #tweezer on
                                ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                                ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(tweezer_slm_amp, t_clock, phase_time_list[j], WithUnit(tweezer_op_amp,"V"))
                                
                                #some strange setting to avoid overshoot
                                # integrator_mitigation_time = np.min([phase_time_list[j],18*1e-6])
                                # ao.setvoltagepulse(tweezer_slm_amp, t_clock, integrator_mitigation_time, WithUnit(2.0,"V"))
                                # if integrator_mitigation_time<phase_time_list[j]:
                                #     ao.setvoltagepulse(tweezer_slm_amp, t_clock+integrator_mitigation_time,phase_time_list[j]-integrator_mitigation_time, WithUnit(0.0,"V"))
                                # ao.setvoltagepulse(tweezer_slm_amp, t_clock+phase_time_list[j]-4e-6, 4e-6, WithUnit(tweezer_recap_amp,"V"))

                                
                                
                                ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))

                                #depreciated bias_yv_for_op_quant_axis as we now use TTL for switching. We keep the nulling amp to make power supply more stable when switching
                                ttl.PulseOn(bias_y_switch_ttl, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                                ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_op_quant_axis,"V")) 
                                if bias_zv_for_op_quant_axis <0:
                                    ttl.PulseOn(biasBz_reverse_ttl, t_clock, phase_time_list[j])
                                else:
                                    ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(np.abs(bias_zv_for_op_quant_axis),"V")) 

                            elif push_out_phase_mode == 'release_recap':
                                #ttl.PulseOn(OP_dds_switch, t_clock, phase_time_list[j])
                                DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'), 
                                            WithUnit(phase_time_list[j], 's'), 
                                            WithUnit(coolingdetune_for_op, 'MHz'),
                                            WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB'))) 
                                                       
                                #no repump
                                #ttl.PulseOn(repump_motAOM, t_clock, push_out_time)

                                #tweezer off
                                #ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                                #ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(tweezer_slm_amp, t_clock, phase_time_list[j], WithUnit(0,"V"))
                                
                                #some strange setting to avoid overshoot
                                # integrator_mitigation_time = np.min([phase_time_list[j],18*1e-6])
                                # ao.setvoltagepulse(tweezer_slm_amp, t_clock, integrator_mitigation_time, WithUnit(2.0,"V"))
                                # if integrator_mitigation_time<phase_time_list[j]:
                                #     ao.setvoltagepulse(tweezer_slm_amp, t_clock+integrator_mitigation_time,phase_time_list[j]-integrator_mitigation_time, WithUnit(0.0,"V"))
                                # ao.setvoltagepulse(tweezer_slm_amp, t_clock+phase_time_list[j]-4e-6, 4e-6, WithUnit(tweezer_recap_amp,"V"))

                                
                                
                                ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))

                                #depreciated bias_yv_for_op_quant_axis as we now use TTL for switching. We keep the nulling amp to make power supply more stable when switching
                                #ttl.PulseOn(bias_y_switch_ttl, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                                ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_nulling,"V")) 
                                if bias_zv_for_op_quant_axis <0:
                                    ttl.PulseOn(biasBz_reverse_ttl, t_clock, phase_time_list[j])
                                else:
                                    ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(np.abs(bias_zv_for_nulling),"V"))

                            elif push_out_phase_mode == 'fastpush':
                                DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'), 
                                            WithUnit(phase_time_list[j], 's'), 
                                            WithUnit(coolingdetune_for_fast_pushout, 'MHz'), 
                                            WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB'))) 
                                DDS.append((OP_dds, WithUnit(t_clock -dds_start_time_delay + phase_time_list[j] - push_out_time -2e-6, 's'),
                                            WithUnit(push_out_time+2e-6, 's'), 
                                            WithUnit(opAOM_freq, 'MHz'), 
                                            WithUnit(push_out_amp_fastpush, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                                #DDS.append((op_repump_dds, WithUnit(t_clock -dds_start_time_delay, 's'),
                                #   WithUnit(push_out_time, 's'),
                                #   WithUnit(op_repump_aom_freq, 'MHz'),
                                #   WithUnit(-11, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                                ttl.PulseOn(OP_dds_switch, t_clock+ phase_time_list[j] - push_out_time-1e-6, push_out_time)                      
                                ttl.PulseOn(repump_motAOM, t_clock+ phase_time_list[j] - push_out_time-1e-6, push_out_time)

                                #ttl.PulseOn(three_d_motAOM, t_clock+(phase_time_list[j]-push_out_time)/2, push_out_time)
                                #tweezer off
                                #ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                                #ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(tweezer_slm_amp, t_clock, phase_time_list[j], WithUnit(0,"V"))
                                
                                #some strange setting to avoid overshoot
                                # integrator_mitigation_time = np.min([phase_time_list[j],18*1e-6])
                                # ao.setvoltagepulse(tweezer_slm_amp, t_clock, integrator_mitigation_time, WithUnit(2.0,"V"))
                                # if integrator_mitigation_time<phase_time_list[j]:
                                #     ao.setvoltagepulse(tweezer_slm_amp, t_clock+integrator_mitigation_time,phase_time_list[j]-integrator_mitigation_time, WithUnit(0.0,"V"))
                                # ao.setvoltagepulse(tweezer_slm_amp, t_clock+phase_time_list[j]-4e-6, 4e-6, WithUnit(tweezer_recap_amp,"V"))

                                
                                
                                ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))

                                #depreciated bias_yv_for_op_quant_axis as we now use TTL for switching. We keep the nulling amp to make power supply more stable when switching
                                #ttl.PulseOn(bias_y_switch_ttl, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                                ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_nulling,"V")) 
                                if bias_zv_for_op_quant_axis <0:
                                    ttl.PulseOn(biasBz_reverse_ttl, t_clock, phase_time_list[j])
                                else:
                                    ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                                ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(np.abs(bias_zv_for_nulling),"V"))


                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)
                        
                        
                        j+=1
                        if phase_time_list[j]>0:
                            ################
                            ###### Phase8: Turn off trap, UV excitation then blow out then UV exicte again
                            #phase6_time = tweezer_off_delay + uv_duration + push_out_time
                            
                            DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'),
                                        WithUnit(phase_time_list[j], 's'),
                                        WithUnit(coolingdetune_for_fast_pushout, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB'))) 
                            
                            if uv_phase_mode == "single_pi_pulse":
                                ttl.PulseOn(red_aom_switch_ttl, t_clock - uv_on_delay - red_high_duration/2, red_high_duration)
                                ttl.PulseOn(uv_aom, t_clock - uv_on_delay , np.max([uv_duration, 3e-7]))
                                
                                #ttl.PulseOn(uv_aom, t_clock - uv_on_delay+ uv_duration+uv_pulse_separation , np.max([uv_duration, 3e-7]))
                                #ttl.PulseOn(uv_aom, t_clock - uv_on_delay+ (uv_duration+uv_pulse_separation)*2 , np.max([uv_duration, 3e-7]))
                                #ttl.PulseOn(uv_aom, t_clock - uv_on_delay+ (uv_duration+uv_pulse_separation)*3 , np.max([uv_duration, 3e-7]))
                                #ttl.PulseOn(uv_aom, t_clock - uv_on_delay+ (uv_duration+uv_pulse_separation)*4 , np.max([uv_duration, 3e-7]))
                                #ttl.PulseOn(uv_aom, t_clock - uv_on_delay+ (uv_duration+uv_pulse_separation)*5 , np.max([uv_duration, 3e-7]))
                                #ttl.PulseOn(uv_aom, t_clock - uv_on_delay+ (uv_duration+uv_pulse_separation)*6 , np.max([uv_duration, 3e-7]))
                                #ttl.PulseOn(uv_aom, t_clock - uv_on_delay+ (uv_duration+uv_pulse_separation)*7 , np.max([uv_duration, 3e-7]))
                                #DDS.append((OP_dds, WithUnit(t_clock -dds_start_time_delay + 2e-6 + uv_duration, 's'), 
                                #            WithUnit(pushout_time_afteruv, 's'),
                                #            WithUnit(opAOM_freq, 'MHz'), 
                                #            WithUnit(push_out_amp_fastpush, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                                
                                
                                #DDS.append((op_repump_dds, WithUnit(t_clock -dds_start_time_delay + 2e-6 + uv_duration, 's'), 
                                #   WithUnit(pushout_time_afteruv, 's'),
                                #   WithUnit(op_repump_aom_freq, 'MHz'),
                                #   WithUnit(-11, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                                                      
                                #ttl.PulseOn(repump_motAOM, t_clock -dds_start_time_delay + 2e-6 + uv_duration, pushout_time_afteruv)



                            if uv_phase_mode == "two_pi_pulse":
                                #pass
                            
                                ttl.PulseOn(red_aom_switch_ttl, t_clock - uv_on_delay - red_high_duration*0.8, red_high_duration)

                                ttl.PulseOn(uv_aom, t_clock - uv_on_delay + 0.3e-6 , np.max([uv_pi_time, 0e-6]))
                                
                                DDS.append((OP_dds, WithUnit(t_clock -dds_start_time_delay - 1e-6  + uv_pi_time, 's'), 
                                            WithUnit(uv_pulse_separation + 4e-6, 's'), 
                                            WithUnit(opAOM_freq, 'MHz'), 
                                            WithUnit(push_out_amp_fastpush, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                                
                                ttl.PulseOn(OP_dds_switch, t_clock + uv_pi_time + 0.8e-6, uv_pulse_separation-1e-6)
                                                      
                                ttl.PulseOn(repump_motAOM, t_clock + uv_pi_time + 0.8e-6, uv_pulse_separation-1e-6)

                                ttl.PulseOn(uv_aom, t_clock - uv_on_delay + uv_pi_time + uv_pulse_separation + 0.3e-6, np.max([uv_pi_time, 0e-6]))

                            

                                                    
                           
                            ao.setvoltagepulse(tweezer_slm_amp, t_clock, phase_time_list[j], WithUnit(0,"V"))
                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))

                            #depreciated bias_yv_for_op_quant_axis as we now use TTL for switching. We keep the nulling amp to make power supply more stable when switching
                            ttl.PulseOn(bias_y_switch_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))

                            ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_op_quant_axis,"V"))
                            
                            ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(np.abs(bias_zv_for_op_quant_axis),"V"))
                            
                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)
                        
                        j+=1
                        if phase_time_list[j]>0:
                            ################
                            ###### Phase9: Turn on tweezer to high depth to recap atoms (loading depth here)
                            #phase8_time = recap_duration

                            DDS.append((offsetlock, WithUnit(t_clock-dds_start_time_delay, 's'), 
                                        WithUnit(phase_time_list[j], 's'), 
                                        WithUnit(coolingdetune_for_img, 'MHz'), 
                                        WithUnit(offsetlock_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB'))) 
                            DDS.append((OP_dds, WithUnit(t_clock -dds_start_time_delay+5e-6, 's'), 
                                        WithUnit(phase_time_list[j]-5e-6, 's'), 
                                        WithUnit(opAOM_shut_freq, 'MHz'), 
                                        WithUnit(opAOM_shut_amp, 'dBm'), WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))  ### @@@?? offsetlock_amp does it vary
                             
                            ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])
                            print("recap phase on")

                            ao.setvoltagepulse(tweezer_slm_amp, t_clock, recap_hold_duration, WithUnit(tweezer_recap_ini_amp,"V"))
                            ao.rampVoltageTime(tweezer_slm_amp,
                                                  WithUnit(tweezer_recap_ini_amp,"V"), WithUnit(tweezer_recap_final_amp,"V"),
                                                  t_clock+recap_hold_duration, t_clock+phase_time_list[j])
                            

                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))


                            ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_nulling,"V")) ##
                            ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                            ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(bias_zv_for_nulling,"V")) ## 


                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)

                        j+=1
                        if phase_time_list[j]>0:
                            ################
                            ###### Phase10: Lower tweezer and apply img with PGC config
                            # assert(delay_after_switching_bfield>=tweezer_ramp_down_time)
                            # phase8_time = delay_after_switching_bfield + coolingimg_time
                            DDS.append((offsetlock, WithUnit(t_clock, 's'),
                                        WithUnit(phase_time_list[j], 's'),
                                        WithUnit(coolingdetune_for_img, 'MHz'),
                                        WithUnit(offsetlock_amp, 'dBm'),
                                        WithUnit(0.0, 'deg'), WithUnit(0, 'MHz'),WithUnit(0, 'dB')))
                            
                            ttl.PulseOn(tweezer_slm_AOM, t_clock, phase_time_list[j])
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, phase_time_list[j])
                            print("phase10on")

                            # ao.rampToVoltageTime(tweezer_slm_amp,WithUnit(tweezer_img_amp,"V"),
                            #                      t_clock, t_clock + tweezer_ramp_down_time)
                            # ao.setvoltagepulse(tweezer_slm_amp, 
                            #                    t_clock + tweezer_ramp_down_time, 
                            #                    phase_time_list[j] - tweezer_ramp_down_time, WithUnit(tweezer_img_amp,"V"))
                            
                            ao.setvoltagepulse(tweezer_slm_amp, 
                                            t_clock , 
                                            phase_time_list[j] , WithUnit(tweezer_img_amp,"V"))
                            
                            for i in range(img_num-1):
                                ttl.PulseOn(three_d_motAOM, t_clock + delay_after_switching_bfield + coolingimg_time*i, exposure_time) ####@changed from pgc_cooling_time*2 to pgc_cooling_time
                                ttl.PulseOn(repump_motAOM, t_clock + delay_after_switching_bfield + coolingimg_time*i, exposure_time)
                                ttl.PulseOn(tweezer_camera_trigger, t_clock + delay_after_switching_bfield-tweezer_cam_delay + coolingimg_time*i, 1e-5)
                            
                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, phase_time_list[j], WithUnit(coolingamp_for_img,"V"))
                            ao.setvoltagepulse(bias_fieldx, t_clock, phase_time_list[j], WithUnit(bias_xv_for_nulling,"V")) ##
                            ao.setvoltagepulse(bias_fieldy, t_clock, phase_time_list[j], WithUnit(bias_yv_for_nulling,"V"))
                            ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, phase_time_list[j], WithUnit(bias_zv_for_nulling,"V")) ## 

                            
                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)


                        j+=1
                        if phase_time_list[j]>0:
                            ################
                            ###### Phase11: Wait for thermal effect on AOMs, should also avoiding defining DDS so that DDS has buffer time to wiat for new trigger
                            #phase9_time = tweezer_wait_time
                            
                            ttl.PulseOn(tweezer_slm_AOM, t_clock, duration -t_clock)
                            ttl.PulseOn(tweezer_slm_aom_pid, t_clock, duration -t_clock)
                            ao.setvoltagepulse(tweezer_slm_amp, t_clock, duration -t_clock, WithUnit(tweezer_waiting_amp,"V"))
                            ao.setvoltagepulse(three_d_motAOM_amp, t_clock, duration -t_clock, WithUnit(coolingamp_for_loading,"V"))

                            ttl.PulseOn(three_d_motAOM, t_clock, duration -t_clock)
                            ttl.PulseOn(repump_motAOM, t_clock, duration -t_clock)

                            ao.setvoltagepulse(bias_fieldx, t_clock, duration -t_clock, WithUnit(bias_xv_for_loading,"V")) ##
                            ao.setvoltagepulse(bias_fieldy, t_clock, duration -t_clock, WithUnit(bias_yv_for_loading,"V"))
                            ttl.PulseOn(biasBz_forward_ttl, t_clock, phase_time_list[j])
                            ao.setvoltagepulse(bias_fieldz, t_clock, duration -t_clock, WithUnit(bias_zv_for_loading,"V")) ## 

                            t_clock += phase_time_list[j]
                            t_clock_list.append(t_clock)


                        

                        #### SET final states to be clear
                        ### Keep the tweezer AOM on to avoid thermal effect
                        
                        ao.setFinalState(efieldvy_ao, WithUnit(efieldvy,"V"))
                        ao.setFinalState(efieldvz_ao, WithUnit(efieldvz,"V"))
                        ao.setFinalState(efieldvx_ao, WithUnit(efieldvx,"V"))
                        ao.setFinalState(tweezer_slm_amp, WithUnit(tweezer_waiting_amp,"V"))  
                        ao.setFinalState(three_d_motAOM_amp, WithUnit(coolingamp_for_loading,"V"))  
                        #ao.setFinalState(offsetlock, WithUnit(coolingdetune_for_loading,"V"))
                        ao.setFinalState(bias_fieldx, WithUnit(bias_xv_for_loading,"V")) 
                        ao.setFinalState(bias_fieldy, WithUnit(bias_yv_for_loading,"V")) 
                        ao.setFinalState(bias_fieldz, WithUnit(bias_zv_for_loading,"V")) 
                        ttl.SetFinalState(biasBz_forward_ttl,1)

                        ttl.SetFinalState(tweezer_slm_AOM,1)
                        ttl.SetFinalState(tweezer_slm_aom_pid,1)
                        ttl.SetFinalState(three_d_motAOM,1)
                        ttl.SetFinalState(repump_motAOM,1)
                        ttl.SetFinalState(two_d_motAOM,1)
                        ttl.SetFinalState(push_beamAOM,1)
                        ttl.SetFinalState(opAOM,0)  
                        #ttl.SetFinalState(mot_coil,1) 

                                #!!!AO must be in front of the TTL
                                ##If we use TTL for the pulse for triggering things, it will need to be in the last
                                #we run this just before the ao.runwaveform (also ttl.runwaveform)
                                #image_file_name = f"release_recapture_debug_{exp_data_collection_counter}"
                                #image_file_name = f"detuning{coolingdetune_for_push_out}_optime{op_pump_time}_deptime{op_depump_time}"
                        # image_file_name = f"uv_freq{uv_freq}uv_duration{uv_duration}Pushout_detuning{coolingdetune_for_push_out:.4f}pushoutime{push_out_time}_optime{op_pump_time}_opdetuning{coolingdetune_for_op:.4f}OPxv{bias_xv_for_op_quant_axis}_zv{bias_zv_for_op_quant_axis}OD2.5_1"
                        image_file_name = datetime.now().strftime("%Y%m%d%H%M%S%f")[:-3]
                        cxn.pvcamNuvuRfsocServer.acquireFastSequence(folder_path ,image_file_name, img_num*repeat_num)


                        #directory, file name (usually use the parameter settings), number of total images = img_num*repeat_num
                        #### need to create an empty folder named pvcam in this folder
                        ### all these images will be save together in a .mat file which is can be loaded as a 3D array
                        pulser.add_dds_pulses(DDS)
                        pulser.program_sequence()
                        pulser.start_number(repeat_num)
                        ao.runWaveform(repeat_num)
                        #while True:
                        ttl.runwaveform(repeat_num)
                        time.sleep(duration*repeat_num)
                        
                        next_counter = 0
                        while(next_counter==0):
                            try:
                                #exp_data_collection_counter = 2
                                # Load the .mat file
                                mat_data = loadmat(folder_path+'\\pvcam\\'+image_file_name+'.mat')
                            except:
                                print("still loading")
                                time.sleep(1)
                                continue
                            next_counter = 1
                        pulser.stop_sequence()

print("total time = ", duration*repeat_num)
#assert(image_delay>exposure_time) ### refresh camera during the delay before taking images

recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on
still loading
recap phase on
phase10on


In [11]:
red_high_duration

0.0012

In [7]:
tweezer_waiting_amp

0.41379310344827586

In [116]:
uv_freq, uv_on_delay

(90000000.0, 2e-06)

In [66]:
efieldvz_ao

9

In [5]:
efieldvy_ao

11

In [43]:
uv_freq = 89.963e6
with labrad.connect() as cxn:  #### 635 to 585MHz
    cxn.windfreakserver.freqhop(1, uv_freq)
    time.sleep(0.5)

In [190]:
uv_freq = 89.963e6
with labrad.connect() as cxn: 
    cxn.windfreakserver.stopOutput(1)
    #cxn.windfreakserver.setSineWave(1, uv_freq,-25) ### -25 makes the signal better for n=60... 507MHz
    #cxn.windfreakserver.setSineWave(1, uv_freq,-25)

In [187]:
#code for DDS
uv_freq = 89.963e6
with labrad.connect() as cxn: 
    
    cxn.pulser.Amplitude("DDS6", WithUnit(-33,"dBm"))
    cxn.pulser.Frequency("DDS6", WithUnit(uv_freq/1e6,"MHz")) ### -25 makes the signal better for n=60... 507MHz

In [193]:
with labrad.connect() as cxn: 
    #cxn.pulser.Amplitude("DDS4", WithUnit(-38,"dBm"))
    cxn.pulser.freqhop("DDS6", WithUnit(89.963,"MHz")) ### -25 makes the signal better for n=60... 507MHz)

In [140]:
uv_phase_mode

'two_pi_pulse'

In [32]:
np.max([uv_release_time,16e-6])

1.6e-05

In [18]:
-dds_start_time_delay + 1e-6 + uv_duration, t_clock - uv_on_delay

(-1.1215e-05, 1.8030660000000003)

In [23]:
if 1==1:
    pass
    print("good")

good


In [36]:
uv_pulse_separation-1e-6

3e-06

In [61]:
op_depump_time

0

In [51]:
repeat_num

20

In [151]:

extra_idling_time_list

array([0.  , 0.01, 0.02, 0.04, 0.06])

In [30]:
cooling_delay_after_trap_on, tweezer_recap_delay

(1e-05, 2e-06)

In [27]:
np.ceil((uv_extra_delay + rydberg_lifetime_duration_list + uv_pi_time*2+red_delay_time + push_out_time)*1e6 / 4) * 4/1e6#uv_pi_time + pi_pulse_separation + uv_pi_time

array([1.2e-05, 1.2e-05, 1.6e-05, 1.6e-05, 2.0e-05, 2.0e-05, 2.4e-05,
       2.4e-05])

In [35]:
phase_time_list*1e6/4

array([2.250000e+05, 1.000000e+04, 2.750000e+04, 1.000000e+03,
       1.250625e+04, 5.000000e+03, 0.000000e+00, 5.250000e-01,
       2.500000e+02, 3.500000e+04, 7.500000e+04])

In [11]:
uv_extra_delay

1.5e-06

In [19]:
with labrad.connect() as cxn:
    repeatf =1 
    durationf = 0.1
    ao = cxn.aoserver
    ttl = cxn.finitedopulses
    ao.blankwaveform(durationf)
    ttl.blankwaveform(durationf)
    ttl.PulseOn(tweezer_slm_AOM, 0, durationf)
    ao.setvoltagepulse(tweezer_slm_amp, 0, durationf, WithUnit(2,"V"))
    
    ao.setFinalState(tweezer_slm_amp, WithUnit(2,"V"))
    ao.runWaveform(repeatf)
                        #while True:
    ttl.runwaveform(repeatf)

In [37]:
coolingdetune_for_push_out_list

array([135.725])

In [10]:
t_clock_list

[0,
 0.9,
 0.9400000000000001,
 1.0,
 1.2000250000000001,
 1.200071,
 1.201071,
 1.341071,
 1.641071]

In [33]:
uv_duration

1.9999999999999998e-05

In [31]:
folder_path

'C:\\Users\\Cryo_rdyberg\\Princeton Dropbox\\Yukai Lu\\CryoRydberg\\Data\\2025\\10\\10\\bg_test_suppose_working'

In [9]:
np.arange(0, 20, 1)*1e-6

array([0.0e+00, 1.0e-06, 2.0e-06, 3.0e-06, 4.0e-06, 5.0e-06, 6.0e-06,
       7.0e-06, 8.0e-06, 9.0e-06, 1.0e-05, 1.1e-05, 1.2e-05, 1.3e-05,
       1.4e-05, 1.5e-05, 1.6e-05, 1.7e-05, 1.8e-05, 1.9e-05])

questions